*❗❗ Before you run this lab, go to Runtime -> Change Runtime Type -> Choose: T4 GPU*

# Module 4 Lab: Investigating the RAG Pipeline
**AIML 2003 — Natural Language Processing**

*One document. One vector database. What does "similar" actually mean?*

---

**This is the standalone NLP lab for Module 4.** You will work in one notebook, give one 3–5 minute demo, and submit your GitHub repo link to Canvas.

## How This Lab Works

This lab is different from previous labs.

In Modules 2 and 3, you built pipelines from scratch — TF-IDF vectors, sentence embeddings, ChromaDB storage. You know how to construct these systems. This week, construction isn't the point.

Part 1 gives you a working RAG pipeline: PDF text extraction, chunking, embedding with a modern sentence transformer, ChromaDB storage, retrieval, and Gemini-powered answer generation. The code is all visible — read it as you run it — but you don't need to write it.

Your job is to **run the system, then investigate it.** Part 2 contains four experiments. Each one asks a question about how the text vector space works, gives you a procedure to answer it, and asks you to record what you found. The experiments build on each other. By the end, you'll understand things about embeddings that you can't learn by building a pipeline — things you can only learn by poking at one and watching what happens.

**For each experiment:** write your code, run it, and then write a markdown cell explaining what you observed and what it means. The markdown cells are not filler. They're the core of your demo.

---
# Part 1: The System

Run these cells in order. Read the code as you go. By the end of Part 1, you'll have a working text RAG pipeline backed by ChromaDB.

**Do not modify Part 1 unless something breaks.** The experiments in Part 2 depend on the variable names and data structures created here.

In [ ]:
# Force Text Wrapping for all outputs
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap !important;
        word-break: break-word !important;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

### Cell 1: Setup and Dependencies

Installs all libraries and imports everything for the text pipeline.

You can disregard dependency errors here if you get an "All imports succeeded" message.

In [ ]:
!pip install -q chromadb sentence-transformers pymupdf google-genai einops

import numpy as np
import matplotlib.pyplot as plt
import fitz
from sentence_transformers import SentenceTransformer
import chromadb
from google import genai
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

print(f"chromadb {chromadb.__version__}")
print("All imports succeeded.")

### Cell 2: Configure Gemini API

You will need to grant access to your Google API key when prompted.

If you get an authentication error, check that the secret is named exactly `GEMINI_API_KEY` and that notebook access is toggled on.


In [ ]:
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Say hello in exactly five words.'
)
print(response.text)

### Cell 3: Load Your Document and Extract Text

Upload a PDF or plain text file (.txt) to Colab (Files panel → Upload). Set `DOC_PATH` below to your filename. Aim for at least 10 pages of selectable text (not scanned images). This cell extracts all text from the file automatically.

You can find great texts for chunking and embedding at [Project Gutenberg](https://www.gutenberg.org/)

**Caching:** The cell copies your file to Google Drive (`/content/drive/MyDrive/AIML_Lab_Cache`) so you don’t have to re-upload in future sessions. If you want a different document later, delete the cached copy from Drive first.

**Cell 3c** lets you inspect chunk boundaries — where one chunk ends and the next begins. With fixed-size chunking, cuts happen mid-sentence. You’ll revisit this in Experiment 4.


In [ ]:
import os
import shutil
import fitz
from google.colab import drive

drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/AIML_Lab_Cache'
os.makedirs(CACHE_DIR, exist_ok=True)

DOC_PATH = "/content/alice.txt"

cached_pdf = os.path.join(CACHE_DIR, "text_source.pdf")
cached_txt = os.path.join(CACHE_DIR, "text_source.txt")

full_text = ""

if os.path.exists(cached_pdf):
    print("Loading cached PDF from Google Drive...")
    doc = fitz.open(cached_pdf)
    for page in doc:
        full_text += page.get_text()
    page_count = doc.page_count
    doc.close()
    print(f"Format: PDF ({page_count} pages)")
elif os.path.exists(cached_txt):
    print("Loading cached plain text from Google Drive...")
    with open(cached_txt, "r", encoding="utf-8") as f:
        full_text = f.read()
    print("Format: Plain text")
else:
    print("No cached document found. Processing uploaded file...")
    if DOC_PATH.lower().endswith(".pdf"):
        doc = fitz.open(DOC_PATH)
        for page in doc:
            full_text += page.get_text()
        page_count = doc.page_count
        doc.close()
        shutil.copy(DOC_PATH, cached_pdf)
        print(f"Format: PDF ({page_count} pages)")
        print("Saved PDF to Google Drive cache.")
    elif DOC_PATH.lower().endswith(".txt"):
        with open(DOC_PATH, "r", encoding="utf-8") as f:
            full_text = f.read()
        shutil.copy(DOC_PATH, cached_txt)
        print("Format: Plain text")
        print("Saved plain text to Google Drive cache.")
    else:
        raise ValueError(f"Unsupported file type: {DOC_PATH}\nUse a .pdf or .txt file.")

print(f"Total characters: {len(full_text):,}")
print(f"Total words: {len(full_text.split()):,}")
print(f"\nFirst 500 characters:\n{full_text[:500]}")

### Why Chunk?

An embedding model has a maximum input length. Nomic Embed handles up to 8,192 tokens (roughly 6,000 words) — far more than older models — but that doesn't mean you should feed it entire documents. A single vector can only encode so much meaning. Feed it a 10-page document and the embedding becomes a blurry average of every topic on every page. Feed it a focused passage and the vector captures one coherent idea.

Chunking splits the document into passages short enough that each embedding represents a single topic or argument. The tradeoff: smaller chunks are more focused but lose context. Larger chunks preserve context but dilute the signal. Part 1 uses fixed-size chunking at 300 words. Experiment 4 revisits this choice.

In [ ]:
def chunk_text_fixed(text, size=300, overlap=0):
    words = text.split()
    chunks = []
    stride = size - overlap
    for i in range(0, len(words), stride):
        chunk = ' '.join(words[i:i+size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

def chunk_text_overlap(text, size=300, overlap=50):
    return chunk_text_fixed(text, size=size, overlap=overlap)

def chunk_text_sentence(text, max_size=300):
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    chunks = []
    current_chunk = []
    current_size = 0
    for sent in sentences:
        sent_words = len(sent.split())
        if current_size + sent_words > max_size and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sent]
            current_size = sent_words
        else:
            current_chunk.append(sent)
            current_size += sent_words
    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')
    return chunks

chunks = chunk_text_fixed(full_text, size=300, overlap=0)

print(f"Chunks: {len(chunks)} total")
print(f"Average words per chunk: {sum(len(c.split()) for c in chunks) / len(chunks):.1f}")
print(f"\nFirst 3 chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ({len(chunk.split())} words) ---")
    print(chunk[:200] + "...")

with open(os.path.join(CACHE_DIR, "chunks_output.txt"), "w", encoding="utf-8") as f:
    for i, chunk in enumerate(chunks):
        f.write(f"=== CHUNK {i} ===\n{chunk}\n\n")
print(f"\nChunks saved to {CACHE_DIR}/chunks_output.txt")

In [ ]:
import random
rand_idx = random.randint(0, len(chunks) - 1)
print(f"Random chunk {rand_idx}:")
print(chunks[rand_idx])

### Cell 4: Embed Text Chunks

Encodes all chunks into 768-dimensional dense vectors using Nomic Embed v1.5. This model uses task-specific prefixes — `"search_document: "` when encoding passages for storage, `"search_query: "` when encoding a query at retrieval time. The prefix tells the model whether it's looking at a document or a question, which improves retrieval accuracy.

Nomic Embed also supports **Matryoshka embeddings**: the first 256 dimensions, or even the first 64, carry a disproportionate share of the information. This lab uses the full 768.

In [ ]:
# Cell 4: Embed text chunks (with Google Drive caching)
import os
import numpy as np

text_emb_path = os.path.join(CACHE_DIR, 'text_embeddings.npy')
text_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)

if os.path.exists(text_emb_path):
    print("Loading text embeddings from Google Drive cache...")
    text_embeddings = np.load(text_emb_path)
else:
    print("Computing text embeddings...")
    prefixed_chunks = ["search_document: " + c for c in chunks]
    text_embeddings = text_model.encode(prefixed_chunks, show_progress_bar=True)
    np.save(text_emb_path, text_embeddings)
    print("Saved text embeddings to Google Drive.")

print(f"Model: nomic-ai/nomic-embed-text-v1.5")
print(f"Text embedding shape: {text_embeddings.shape}")
print(f"Dimensionality: {text_embeddings.shape[1]}")
density = np.count_nonzero(text_embeddings) / text_embeddings.size * 100
print(f"Density: {density:.1f}%")


### Cell 5: Build ChromaDB Text Collection

Before you run this cell, it's worth understanding what ChromaDB is and why you need it.

**Traditional databases** store rows of structured data. You query with exact matches: `WHERE text LIKE '%artificial intelligence%'`. A **vector database** stores embedding vectors and retrieves by similarity: give me the 3 nearest vectors to this query vector. SQL matches exact words. Vectors match meaning.

In [ ]:
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("text_chunks")
except Exception:
    pass

text_collection = chroma_client.create_collection(name="text_chunks")
text_collection.add(
    ids=[f"text_{i}" for i in range(len(chunks))],
    embeddings=text_embeddings.tolist(),
    documents=chunks,
    metadatas=[{"chunk_index": i} for i in range(len(chunks))]
)

print(f"Text collection: {text_collection.count()} items, {text_embeddings.shape[1]}d")

In [ ]:
import random

random_text_idx = random.randint(0, len(chunks) - 1)

sample_text = text_collection.get(ids=[f"text_{random_text_idx}"], include=["embeddings", "documents", "metadatas"])

print("=== One record from the text collection ===")
print(f"  ID:        {sample_text['ids'][0]}")
print(f"  Document:  {sample_text['documents'][0][:120]}...")
print(f"  Metadata:  {sample_text['metadatas'][0]}")
text_emb = sample_text['embeddings'][0]
print(f"  Embedding (first 40 dims): {[round(float(x), 4) for x in text_emb[:40]]}")
print(f"  ... ({len(text_emb)} dimensions total)")

print()
print("In a SQL database: WHERE text LIKE '%artificial intelligence%'")
print("In vector database: give me the 3 nearest vectors to this query vector")
print("SQL matches exact words. Vectors match meaning.")

### Cell 6: Search Functions and RAG Pipeline

Defines text search and the `ask_document` RAG function that retrieves context and passes it to Gemini.

The RAG system prompt instructs Gemini to use only the provided context. If the answer isn’t in the context, it should say so rather than guessing.


In [ ]:
def search_text(query, n=3):
    prefixed = "search_query: " + query
    query_embedding = text_model.encode([prefixed]).tolist()
    return text_collection.query(query_embeddings=query_embedding, n_results=n)

def ask_document(question, n_passages=3):
    results = search_text(question, n=n_passages)
    passages = results["documents"][0]
    distances = results["distances"][0]

    context = "\n\n---\n\n".join(passages)
    prompt = f"""You are a document assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I cannot find this in the document."
Do not guess or add information beyond what is provided.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )

    return {
        "question": question,
        "retrieved_passages": passages,
        "distances": distances,
        "answer": response.text
    }

print("=== Text RAG Test ===")
result = ask_document("Summarize this text.")
print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print(f"Top passage distance: {result['distances'][0]:.4f}")

---

**Part 1 checkpoint.** You should now have a working text RAG pipeline backed by ChromaDB. If anything above threw an error, fix it before continuing — every experiment in Part 2 depends on these variables and functions.

---

# Part 2: Experiments

Four experiments, each asking a different question about how the text vector space works. Run the code, record what you find, and write the markdown reflection for each one. The reflections are the substance of your demo.

## Experiment 1: The Similarity Game

**The question:** What makes a "good" query for a sentence embedding model? When you write a query by hand, what strategies produce the highest cosine similarity to a target passage?

**The procedure:**

1. Run the cell below to pick a random passage from your document and display it.
2. Read the passage carefully.
3. Write a query sentence — anything you think will score high similarity against that passage.
4. The cell scores your query and logs it.
5. Repeat at least five times. Try different strategies: exact vocabulary, synonyms, paraphrasing, short vs. long, abstract vs. concrete.
6. After your attempts, answer the questions in the markdown cell.

In [ ]:
import random

# Pick a target passage (change the index to try different ones)
TARGET_INDEX = random.randint(0, len(chunks) - 1)

##################################################
#Open chunks_output.txt in your Google Drive if you want to select a *specific* chunk.
#Change chunkNumber below and uncomment the two lines below.
#chunkNumber = 12
#TARGET_INDEX = chunkNumber
##################################################

target_passage = chunks[TARGET_INDEX]
target_embedding = text_embeddings[TARGET_INDEX]

print("=" * 60)
print("YOUR TARGET PASSAGE")
print("=" * 60)
print(target_passage[:500])
if len(target_passage) > 500:
    print(f"\n... ({len(target_passage)} chars total)")
print("=" * 60)


In [ ]:
# Score your queries here. Run this cell once per attempt.
# Change the query string each time and re-run.

###################################################
# Change this query ⬇️ every time you run this cell
MY_QUERY = "What is the main topic of this passage?"
###################################################

# Nomic needs the query prefix for fair comparison
query_embedding = text_model.encode(["search_query: " + MY_QUERY])
sim = cosine_similarity(query_embedding, target_embedding.reshape(1, -1))[0][0]

# Log the attempt
if 'text_attempts' not in dir():
    text_attempts = []
text_attempts.append({"query": MY_QUERY, "similarity": float(sim)})

print(f"Attempt {len(text_attempts)}: {sim:.4f}")
print(f"Query: {MY_QUERY}")

# Show all attempts so far
print(f"\n{'='*60}")
print(f"{'#':<4} {'Score':<10} Query")
print(f"{'='*60}")
for i, a in enumerate(text_attempts):
    marker = " ← best" if a['similarity'] == max(x['similarity'] for x in text_attempts) else ""
    print(f"{i+1:<4} {a['similarity']:<10.4f} {a['query'][:60]}{marker}")


**✍️ Experiment 1 Reflection**

1. What was your highest similarity score? What strategy produced it?
2. Did copying exact words from the passage score higher than capturing the overall meaning in different words? By how much?
3. At what point did your scores stop improving? What does that ceiling tell you about what the model encodes?

## Experiment 2: Embedding Surgery

**The question:** Embedding vectors aren't just opaque numbers — they have geometric structure. If you average two embeddings, does the midpoint land somewhere meaningful?

**The procedure:**

1. Pick two text chunks on different topics. Average their embeddings. Retrieve the closest passage to the midpoint.
2. Try it with three chunks on three different topics.
3. Try averaging two chunks on the *same* topic. Does the midpoint retrieve something "more typical"?
4. Try subtracting: take chunk A's embedding, subtract chunk B's embedding, and retrieve the nearest passage. What does the "difference vector" point to?

In [ ]:
# Embedding surgery
# Look in /content/drive/MyDrive/AIML_Lab_Cache/chunks_output.txt
# Pick two chunks on different topics

###################################################
# Change  ⬇️ this number to the chunk you want to use
CHUNK_A = 0   # ← change to a chunk about topic A

CHUNK_B = 10  # ← change to a chunk about a different topic
###################################################

print("CHUNK A (first 200 chars):")
print(chunks[CHUNK_A][:200])
print(f"\nCHUNK B (first 200 chars):")
print(chunks[CHUNK_B][:200])

# Average the embeddings
midpoint = (text_embeddings[CHUNK_A] + text_embeddings[CHUNK_B]) / 2.0

# Retrieve closest passage to the midpoint (fetch extra to manually filter)
mid_results = text_collection.query(
    query_embeddings=[midpoint.tolist()],
    n_results=10
)

print(f"\n{'='*60}")
print("NEAREST PASSAGES TO THE MIDPOINT (excluding A & B)")
print(f"{'='*60}")

count = 0
for meta, doc, dist in zip(mid_results['metadatas'][0], mid_results['documents'][0], mid_results['distances'][0]):
    if meta['chunk_index'] in [CHUNK_A, CHUNK_B]:
        continue
    print(f"\n#{count+1} (distance: {dist:.4f}):")
    print(doc[:200])
    count += 1
    if count >= 5:
        break


**✍️ Experiment 2 Reflection**

1. For the text midpoint: did the retrieved passage relate to both source topics, to one of them, or to something else entirely?
2. What happened when you averaged two chunks from the same topic? Did the midpoint retrieve something "more typical"?

## Experiment 3: The Caption Bridge

**The question:** Your vector database indexes text. But what if you wanted to search it starting from an image? Gemini Vision can caption an image, and that caption can be embedded as text and used to search the text collection. How much information survives this translation?

**The procedure:**

1. Run the setup cell below to load a small set of sample images.
2. Run the caption bridge on five images from different classes.
3. For each image, examine three things: what the image actually shows, what Gemini's caption says, and what the text retrieval finds using that caption.
4. Rate each step of the chain: did the caption capture the image's content? Did the text retrieval find something relevant to the caption?
5. Connect this back to Experiment 1: is Gemini writing "good queries" by the standards you discovered?

In [ ]:
import os, pathlib
import tensorflow as tf
from PIL import Image as PILImage
from google.colab import drive

drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/AIML_Lab_Cache'
os.makedirs(CACHE_DIR, exist_ok=True)

DATASET = "flowers"

cached_images_path = os.path.join(CACHE_DIR, f"{DATASET}_images_bridge.npz")

if os.path.exists(cached_images_path):
    print(f"Loading cached {DATASET} images from Google Drive...")
    data = np.load(cached_images_path, allow_pickle=True)
    bridge_images = list(data['images'])
    bridge_labels = list(data['labels'])
else:
    print(f"Downloading {DATASET} dataset for caption bridge...")
    data_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
    extracted_path = tf.keras.utils.get_file("flower_photos", origin=data_url, untar=True)
    data_dir = pathlib.Path(extracted_path)
    if (data_dir / "flower_photos").exists():
        data_dir = data_dir / "flower_photos"
    license_file = data_dir / "LICENSE.txt"
    if license_file.exists():
        license_file.unlink()

    bridge_images = []
    bridge_labels = []
    N_PER_CLASS = 5

    for class_dir in sorted(d for d in data_dir.iterdir() if d.is_dir()):
        class_name = class_dir.name
        image_files = sorted(class_dir.glob("*.jpg"))[:N_PER_CLASS]
        for img_path in image_files:
            try:
                img = PILImage.open(img_path).convert("RGB").resize((224, 224))
                bridge_images.append(np.array(img))
                bridge_labels.append(class_name)
            except Exception:
                continue

    np.savez(cached_images_path,
             images=np.array(bridge_images),
             labels=np.array(bridge_labels))

print(f"Loaded {len(bridge_images)} images across {len(set(bridge_labels))} classes")
print(f"Classes: {sorted(set(bridge_labels))}")

classes = sorted(set(bridge_labels))
fig, axes = plt.subplots(1, len(classes), figsize=(3 * len(classes), 3))
if len(classes) == 1:
    axes = [axes]
for i, cls in enumerate(classes):
    idx = next(j for j, l in enumerate(bridge_labels) if l == cls)
    axes[i].imshow(bridge_images[idx])
    axes[i].set_title(cls)
    axes[i].axis('off')
plt.suptitle("Sample images for caption bridge")
plt.tight_layout()
plt.show()

In [ ]:
import io
import base64
from PIL import Image as PILImage

def caption_image(img_array):
    pil_img = PILImage.fromarray(img_array.astype(np.uint8))
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG')
    img_bytes = buf.getvalue()
    b64 = base64.b64encode(img_bytes).decode('utf-8')

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[
            {"text": "Describe this image in one detailed sentence. Be specific about what you see — colors, objects, actions, context."},
            {"inline_data": {"mime_type": "image/jpeg", "data": b64}}
        ]
    )
    return response.text.strip()

def caption_bridge(img_idx):
    caption = caption_image(bridge_images[img_idx])
    caption_embedding = text_model.encode(["search_query: " + caption])
    text_results = text_collection.query(
        query_embeddings=caption_embedding.tolist(),
        n_results=3
    )
    return {
        "image_idx": img_idx,
        "label": bridge_labels[img_idx],
        "caption": caption,
        "retrieved_passages": text_results["documents"][0],
        "distances": text_results["distances"][0]
    }

all_classes = sorted(set(bridge_labels))
bridge_test_classes = all_classes[:5]
BRIDGE_INDICES = []
for cls in bridge_test_classes:
    cls_idx = next(j for j, l in enumerate(bridge_labels) if l == cls)
    BRIDGE_INDICES.append(cls_idx)

print(f"Bridge test images: {[(idx, bridge_labels[idx]) for idx in BRIDGE_INDICES]}\n")

bridge_results = []

for idx in BRIDGE_INDICES:
    result = caption_bridge(idx)
    bridge_results.append(result)

    print(f"\n{'='*60}")
    print(f"IMAGE: idx={idx}, class={result['label']}")
    print(f"{'='*60}")

    fig, ax = plt.subplots(1, 1, figsize=(3, 3))
    ax.imshow(bridge_images[idx])
    ax.set_title(f"{result['label']} (idx={idx})")
    ax.axis('off')
    plt.show()

    print(f"CAPTION: {result['caption']}")
    print(f"\nTOP RETRIEVED PASSAGE (distance: {result['distances'][0]:.4f}):")
    print(result['retrieved_passages'][0][:300])

**✍️ Experiment 3 Reflection**

1. Where did the chain break most often — in the caption step (Gemini misread the image) or the retrieval step (the caption didn't match any passage)?
2. Connect this to Experiment 1: is Gemini writing "good queries" by the standards you discovered? Would a shorter or longer caption have performed better?

## Experiment 4: Chunking Strategy Showdown

**The question:** You used fixed-size chunking to build the vector database in Part 1. But chunking strategy affects everything downstream. Does it actually matter which strategy you use? And for overlap chunking, how much overlap is the right amount?

**The procedure:**

1. Write three questions your RAG pipeline can answer well (you tested these in Part 1).
2. Run the cell below to compare fixed, overlap, and sentence-boundary chunking on those questions.
3. Then run the overlap sweep: try different overlap amounts (25, 50, 100, 150 words) and see how overlap size affects chunk count and retrieval distance. Find the sweet spot for your document.
4. In the final cell, pick your best configuration and run a full RAG answer comparison against the default. Did the answer actually improve?

In [ ]:
test_questions = [
    "What is the main topic?",
    "Who are the main characters?",
    "What are the key events?"
]

strategies = {
    "Fixed (300)": chunk_text_fixed(full_text, size=300, overlap=0),
    "Overlap (300, 50)": chunk_text_overlap(full_text, size=300, overlap=50),
    "Sentence": chunk_text_sentence(full_text, max_size=300)
}

results_by_strategy = {}

for strategy_name, chunks_for_strat in strategies.items():
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name}")
    print(f"Total chunks: {len(chunks_for_strat)}")
    print(f"{'='*60}")

    prefixed_strat = ["search_document: " + c for c in chunks_for_strat]
    embeddings_strat = text_model.encode(prefixed_strat, show_progress_bar=True, convert_to_numpy=True)

    try:
        chroma_client.delete_collection(f"temp_{strategy_name.replace(' ', '_')}")
    except:
        pass

    temp_collection = chroma_client.create_collection(name=f"temp_{strategy_name.replace(' ', '_')}")
    temp_collection.add(
        ids=[f"{strategy_name}_{i}" for i in range(len(chunks_for_strat))],
        embeddings=embeddings_strat.tolist(),
        documents=chunks_for_strat,
        metadatas=[{"chunk_index": i} for i in range(len(chunks_for_strat))]
    )

    for q in test_questions:
        q_prefixed = "search_query: " + q
        q_emb = text_model.encode([q_prefixed])
        q_results = temp_collection.query(query_embeddings=q_emb.tolist(), n_results=1)
        distance = q_results["distances"][0][0]
        print(f"  Q: '{q}' -> distance {distance:.4f}")

    results_by_strategy[strategy_name] = temp_collection

In [ ]:
import matplotlib.pyplot as plt

OVERLAP_VALUES = [0, 25, 50, 100, 150]  # ← add or change values to explore
overlap_data = {}

test_q = test_questions[0]

print(f"{'Overlap':>8} {'Chunks':>8} {'Distance':>10}")
print('=' * 30)

for ovlp in OVERLAP_VALUES:
    chunks_ovlp = chunk_text_overlap(full_text, size=300, overlap=ovlp) if ovlp > 0 else chunk_text_fixed(full_text, size=300)
    prefixed = ["search_document: " + c for c in chunks_ovlp]
    embeddings = text_model.encode(prefixed, show_progress_bar=True)

    q_emb = text_model.encode(["search_query: " + test_q])
    q_sim = cosine_similarity(q_emb, embeddings)
    best_idx = q_sim.argmax()
    best_distance = 1.0 - q_sim[0, best_idx]

    overlap_data[ovlp] = {
        "n_chunks": len(chunks_ovlp),
        "best_distance": best_distance,
        "best_passage": chunks_ovlp[best_idx]
    }
    print(f"{ovlp:>8} {len(chunks_ovlp):>8} {best_distance:>10.4f}")

# Plot the results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: chunk count vs overlap
ax1.plot(OVERLAP_VALUES, [overlap_data[o]['n_chunks'] for o in OVERLAP_VALUES],
         'o-', color='#1a3a5c', linewidth=2, markersize=8)
ax1.set_xlabel('Overlap (words)')
ax1.set_ylabel('Number of chunks')
ax1.set_title('Storage cost: more overlap = more chunks')
ax1.grid(True, alpha=0.3)

# Right: retrieval distance vs overlap
ax2.plot(OVERLAP_VALUES, [overlap_data[o]['best_distance'] for o in OVERLAP_VALUES],
         'o-', color='#2e7d32', linewidth=2, markersize=8)
ax2.set_xlabel('Overlap (words)')
ax2.set_ylabel('Retrieval distance (lower = better)')
ax2.set_title('Retrieval quality vs. overlap')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nLook at the right plot. Does distance keep improving with more overlap,')
print('or does it plateau? Where is the sweet spot for your document?')


In [ ]:
BEST_OVERLAP = 50

best_chunks = chunk_text_overlap(full_text, size=300, overlap=BEST_OVERLAP)
best_prefixed = ["search_document: " + c for c in best_chunks]
best_embeddings = text_model.encode(best_prefixed, show_progress_bar=False, convert_to_numpy=True)

try:
    chroma_client.delete_collection("best_strategy")
except:
    pass

best_collection = chroma_client.create_collection(name="best_strategy")
best_collection.add(
    ids=[f"best_{i}" for i in range(len(best_chunks))],
    embeddings=best_embeddings.tolist(),
    documents=best_chunks,
    metadatas=[{"chunk_index": i} for i in range(len(best_chunks))]
)

def ask_with_collection(collection, question, n_passages=3):
    results = collection.query(query_embeddings=text_model.encode([f"search_query: {question}"]).tolist(), n_results=n_passages)
    passages = results["documents"][0]
    context = "\n\n---\n\n".join(passages)
    prompt = f"""You are a document assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I cannot find this in the document."
Do not guess or add information beyond what is provided.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
    response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
    return response.text

test_q_final = test_questions[0]
print(f"Question: {test_q_final}\n")

print("=== Default (fixed-size, no overlap) ===")
default_answer = ask_document(test_q_final)
print(default_answer["answer"])

print(f"\n=== Best configuration (overlap={BEST_OVERLAP}) ===")
best_answer = ask_with_collection(best_collection, test_q_final)
print(best_answer)

print("\nDid the answer change? Was it better?")

**✍️ Experiment 4 Reflection**

1. In your overlap sweep, where was the sweet spot? Did distance keep improving with more overlap, or did it plateau? At what point did the extra chunks stop being worth it?
2. Did the RAG answer actually change between default and best configuration? If yes, was the new answer better? If no, what does that tell you about how sensitive RAG is to chunking strategy?
3. Sentence-boundary chunking tries to keep sentences intact. Did it outperform fixed-size chunking on your questions, or did the simpler strategy hold up?
4. Context-aware chunking — where an LLM or similarity detector identifies topic boundaries — would go further than any of these mechanical strategies. Based on your results, where do you think mechanical chunking is "good enough" and where would context-awareness help?

# Demo Prep

Your notebook is your demo. No separate slides or written reflection needed.

In three to five minutes, walk the class through:

1. **One experiment in depth.** Pick the experiment that surprised you most. Show the code, show the output, and explain what you learned. This is the centerpiece.
2. **A failure.** Show something that didn't work the way you expected — a bad query, a hallucination, a chunking result that hurt instead of helped. Explain why you think it failed.
3. **The RAG pipeline in action.** Ask your document one question live. Show the retrieved passages and explain why the system found them.
4. **One sentence you couldn't have said before this lab.** What do you now understand about embeddings, vector search, or chunking that you didn't before?

The markdown cells you wrote throughout the lab are your reflection. The demo is your presentation. Come ready to run cells live and talk about what they show.